# 05 · Cenários de incidente (v4)
**CRISP-DM:** Preparação dos Dados (gabarito de avaliação)

Cada **cenário** simula um incidente: uma degradação começa em t₀ e o incidente é registrado em t = t₀ + atraso. A pergunta é respondida **no instante t**, olhando só para trás.

| Motivo | Abrangência | Baixa | Média | Alta | Muito alta |
|---|---|---|---|---|---|
| Erro de negócio (4xx) | generalizado | 5% | 15% | 40% | — |
| Erro de negócio (4xx) | 1 consumidor | 30% | 60% | 90% | — |
| Erro de sistema (5xx) | generalizado | 5% | 15% | 40% | — |
| Erro de sistema (5xx) | 1 consumidor | 30% | 60% | 90% | — |
| Latência | generalizada | ×1,5 | ×2,5 | ×4 | **×10** |

**Latência ×10.** Como a latência vem de médias por minuto, poucas requisições lentas elevam muito o valor: no histórico, o p95 horário passa de 10× a mediana do seu contexto numa pequena fração das horas normais (notebook 06 mostra a distribuição). Degradações de latência relevantes para um incidente tendem a ser desse tamanho.

- **Teste:** 01–20/09, com os 14 recursos selecionados (todos com dados completos de junho a setembro).
- **Atraso** entre o início da degradação e o registro: 15, 30, 60 ou 120 min; **5 repetições** por combinação; o mesmo número de **consultas de controle** (sem falha).
- O instante do registro é sorteado entre 01/09 às 04h e o fim dos dados (20/09).

In [1]:
%run ./00_funcoes.ipynb

00_funcoes (v4) carregado · /home/wsl/gh/mestrado/claude9/v4


In [2]:
brutos = pd.read_parquet(RESULTADOS / "02_brutos_consolidados.parquet")
B = Brutos(brutos)
eleg = elegiveis(B)
eleg.assign(recurso=eleg["recurso"].map(rot)).pivot(index="recurso", columns="mes", values=["dias_hist_1m", "elegivel"]).sort_index()

,dias_hist_1m,elegivel
mes,set,set
recurso,,
A01,31,True
A02,31,True
A03,31,True
A04,31,True
A05,31,True
A06,31,True
A07,31,True
A08,31,True


In [3]:
cenarios = gerar_cenarios(eleg)
cenarios.to_csv(RESULTADOS / "05_cenarios.csv", index=False)
eleg.to_csv(RESULTADOS / "05_elegiveis.csv", index=False)
print(cenarios.groupby(["mes", "tipo"]).size().unstack().to_dict())
cenarios.assign(recurso=cenarios["recurso"].map(rot)).query("tipo == 'incidente'").groupby(["recurso", "mes"]).size().unstack()

{'controle': {'set': 4480}, 'incidente': {'set': 4480}}


mes,set
recurso,
A01,320
A02,320
A03,320
A04,320
A05,320
A06,320
A07,320
A08,320
A09,320


## Exemplo: efeito de um cenário nas faixas antes do registro

In [4]:
ex = cenarios[(cenarios["tipo"] == "incidente") & (cenarios["motivo"] == "5xx_geral") & (cenarios["intensidade"] == "media")
              & (cenarios["atraso"] == 30)].iloc[0]
sem = ex.copy()
sem["tipo"] = "controle"
com_falha = pd.DataFrame(periodos_do_cenario(B, ex))
sem_falha = pd.DataFrame(periodos_do_cenario(B, sem))
cols = ["periodo", "lo", "hi", "vol", "tx_4xx", "tx_5xx", "lat_p95", "n_cli_5xx"]
print("recurso", rot(ex["recurso"]), "· registro", ex["t_registro"], "· início real 30 min antes")
display(sem_falha[cols].round(3).rename(columns=lambda c: c + " (sem)" if c not in ("periodo", "lo", "hi") else c)
        .join(com_falha[["tx_5xx", "n_cli_5xx"]].round(3).rename(columns=lambda c: c + " (com)")))

recurso A12 · registro 2026-09-09 20:17:00 · início real 30 min antes


,periodo,lo,hi,vol (sem),tx_4xx (sem),tx_5xx (sem),lat_p95 (sem),n_cli_5xx (sem),tx_5xx (com),n_cli_5xx (com)
0,L,0,15,3362,0.841,0.001,131.0,1,0.028,2
1,L,0,30,6658,0.830,0.004,129.0,1,0.030,2
2,L,0,60,13706,0.801,0.006,126.0,1,0.019,2
3,banda,0,15,3362,0.841,0.001,131.0,1,0.028,2
4,banda,15,30,3296,0.820,0.008,122.0,1,0.033,1
5,banda,30,60,7048,0.774,0.007,122.0,1,0.007,1
6,banda,60,120,11156,0.790,0.004,136.0,1,0.004,1
7,banda,120,180,11863,0.792,0.005,134.0,1,0.005,1
8,banda,180,240,12770,0.792,0.006,138.0,1,0.006,1
